# Плёнка Ферми и теорема Лиувилля: проверка поправки $F_{12}$

Добавлено 6 нояб. 2026. Вопрос: сохраняет ли динамика с поправкой плёнки Ферми фазовый объём и выводится ли она из действия? Если нет, то детальное равновесие и ФДТ для неё не гарантированы (см. обсуждение «а если микродинамика не гамильтонова?»).

**Исходный закон** (`NOTE_F12_nonsync_fermi_correction_2026-10-25.md`, первое приближение):
$$m_i\,\mathbf a_i=\sum_{j\ne i}\frac{k q_iq_j\,\mathbf r_{ij}}{r_{ij}^3}\left[1+\frac{(\mathbf a_i-\mathbf a_j)\cdot\mathbf r_{ij}}{2c^2}\right].$$

Ускорения входят и справа, поэтому это линейная система на $\{\mathbf a_i\}$ при заданных координатах. Её решение $\mathbf a_i(\mathbf X)$ зависит только от координат, не от скоростей.

**Что проверяем** (обезразмерено $k=1$; $c^2$ — параметр):
1. Лиувилль: $\nabla\cdot\dot{\mathbf Y}$ в фазовом пространстве $\mathbf Y=(\mathbf X,\mathbf V)$.
2. Сохранение импульса: $\sum_i m_i\mathbf a_i=0$.
3. Условие Гельмгольца: сила $m_i\mathbf a_i(\mathbf X)$, не зависящая от скоростей, выводится из лагранжиана $L=T-U(\mathbf X)$ тогда и только тогда, когда матрица $\partial(m_i a_i^\alpha)/\partial X_j^\beta$ симметрична, то есть поле безвихревое.
4. Работа силы по замкнутому контуру: она равна нулю тогда и только тогда, когда энергия сохраняется.

In [ ]:
preparser(False)
import numpy as np
k = 1.0
def accel(X, m, q, c2):
    """Ускорения из m_i a_i = Σ_j k q_i q_j r_ij/r³ · [1 + (a_i − a_j)·r_ij/(2c²)] (неявно по a)."""
    n, d = X.shape; A = np.zeros((n*d, n*d)); b = np.zeros(n*d)
    for i in range(n):
        A[i*d:(i+1)*d, i*d:(i+1)*d] += m[i]*np.eye(d)
        for j in range(n):
            if i == j: continue
            r = X[i]-X[j]; R = np.linalg.norm(r); f = k*q[i]*q[j]*r/R**3
            b[i*d:(i+1)*d] += f
            C = np.outer(f, r)/(2*c2)
            A[i*d:(i+1)*d, i*d:(i+1)*d] -= C; A[i*d:(i+1)*d, j*d:(j+1)*d] += C
    return np.linalg.solve(A, b).reshape(n, d)
def force_jac(X, m, q, c2, h=1e-6):
    n, d = X.shape; N = n*d; J = np.zeros((N, N))
    F = lambda Y: (m[:, None]*accel(Y.reshape(n, d), m, q, c2)).ravel()
    for p in range(N):
        e = np.zeros(N); e[p] = h
        J[:, p] = (F(X.ravel()+e) - F(X.ravel()-e))/(2*h)
    return J
print('модель загружена')

## 1. Лиувилль

$\dot{\mathbf X}=\mathbf V$, $\dot{\mathbf V}=\mathbf a(\mathbf X)$. Значит,
$$\nabla\cdot\dot{\mathbf Y}=\sum\frac{\partial V}{\partial X}+\sum\frac{\partial a(\mathbf X)}{\partial V}=0+0=0$$
**тождественно**, при любом $\mathbf a(\mathbf X)$, консервативном или нет. Ниже проверяется численно: след якобиана полного потока.

In [ ]:
rng = np.random.default_rng(0)
m = np.array([1.0, 2.0, 0.5]); q = np.array([1.0, 1.0, -1.0]); c2 = 1.0
X = rng.normal(size=(3, 2))*3; V = rng.normal(size=(3, 2))
def flow(Y):
    n = 3; Xs = Y[:2*n].reshape(n, 2); Vs = Y[2*n:].reshape(n, 2)
    return np.r_[Vs.ravel(), accel(Xs, m, q, c2).ravel()]
Y = np.r_[X.ravel(), V.ravel()]; h = 1e-6
tr = sum((flow(Y+h*np.eye(12)[p])[p] - flow(Y-h*np.eye(12)[p])[p])/(2*h) for p in range(12))
print(f'div потока = {tr:.2e} ->', 'OK' if abs(tr) < 1e-8 else 'FAIL')

## 2. Импульс и условие Гельмгольца

Для двух тел $\mathbf a_1-\mathbf a_2$ — единственная переменная, и сила центральная. Ожидаем симметричный якобиан (потенциал существует).

Для трёх и более тел множитель $[1+(\mathbf a_i-\mathbf a_j)\cdot\mathbf r_{ij}/2c^2]$ в силе $i\leftarrow j$ зависит через $\mathbf a_i$ от **всех** остальных тел. Сила перестаёт быть градиентом. Мера асимметрии — $\|J-J^T\|/\|J\|$; ожидаем, что она пропорциональна $1/c^2$.

In [ ]:
for n in (2, 3, 4):
    mm = np.array([1.0, 2.0, 0.5, 1.5][:n]); qq = np.array([1.0, 1.0, -1.0, -1.0][:n])
    XX = rng.normal(size=(n, 2))*3
    P = (mm[:, None]*accel(XX, mm, qq, 1.0)).sum(0)
    J = force_jac(XX, mm, qq, 1.0); asym = np.abs(J-J.T).max()/np.abs(J).max()
    print(f'N={n}: Σ m a = {np.abs(P).max():.1e},  асимметрия якобиана = {asym:.2e}')
    if n == 2: ok2 = asym < 1e-8
    if n == 3: X3, m3, q3 = XX, mm, qq
print('N=2 потенциальна:', 'OK' if ok2 else 'FAIL')
sc = [np.abs((J:=force_jac(X3, m3, q3, cc))-J.T).max()/np.abs(J).max() for cc in (1e2, 1e3, 1e4)]
print('N=3: асимметрия при c² = 1e2, 1e3, 1e4:', ['%.2e' % s for s in sc],
      '-> ∝1/c²:', 'OK' if abs(sc[0]/sc[1]-10) < 1 and abs(sc[1]/sc[2]-10) < 1 else 'FAIL')

## 3. Работа по замкнутому контуру

Тела 1 и 2 удерживаются на месте (связи работы не совершают), тело 3 обходит окружность. Если сила консервативна, $W=0$. Здесь $W\ne0$ означает, что за цикл энергия **рождается или исчезает**: это нарушение первого начала, а не второго.

In [ ]:
def W_loop(X, m, q, c2, R0=0.6, i=2, Nt=4001):
    th = np.linspace(0, 2*np.pi, Nt); W = 0.0; c0 = X[i].copy()
    for t0, t1 in zip(th[:-1], th[1:]):
        tm = (t0+t1)/2; Xc = X.copy(); Xc[i] = c0 + R0*np.array([np.cos(tm), np.sin(tm)])
        W += (m[i]*accel(Xc, m, q, c2)[i]) @ (R0*np.array([-np.sin(tm), np.cos(tm)])*(t1-t0))
    return W
X0 = np.array([[0, 0], [4., 0], [1., 3.]]); m0 = np.array([1.0, 2.0, 0.5]); q0 = np.array([1.0, 1.0, -1.0])
Ws = {cc: W_loop(X0, m0, q0, cc) for cc in (1e2, 1e3, 1e4)}
W2 = W_loop(X0[:2], m0[:2], q0[:2], 1.0, i=1)
print('N=3: W при c² = 1e2, 1e3, 1e4:', ['%.3e' % w for w in Ws.values()])
print(f'N=2 (контроль): W = {W2:.1e} ->', 'OK' if abs(W2) < 1e-9 else 'FAIL')
print('N=3: W ≠ 0 и ∝1/c²:', 'OK' if abs(Ws[1e2]/Ws[1e3]-10) < 1 and abs(Ws[1e4]) > 1e-12 else 'FAIL')

## 4. Выводы

**Что следует из расчёта:**
1. **Лиувилль выполняется тождественно.** Ускорение зависит только от координат, поэтому фазовый объём сохраняется при любом $F_{12}$. Тот путь к нарушению детального равновесия, который требует сжатия фазового объёма, $F_{12}$ **не даёт**.
2. **Импульс сохраняется**: сумма сил равна нулю, третий закон для пары выполнен.
3. **Для $N=2$ закон потенциален.** Для $N\ge3$ условие Гельмгольца нарушено в порядке $1/c^2$. Закон в форме первого приближения **не выводится из действия** вида $T-U(\mathbf X)$. Это расходится с утверждением, что $F_{12}$ получена варьированием действия с плёнкой Ферми: интерполяционная формула этому требованию не удовлетворяет.
4. **Работа по замкнутому контуру $\ne0$** при $N\ge3$, порядок $1/c^2$. Энергия не сохраняется, то есть нарушается первое начало, а не второе. При конечной температуре такая система не придёт в равновесие с термостатом — она будет нагревать или охлаждать себя сама.

**Как это понимать.** Пункт 4 — признак неполноты приближения, а не новой физики. Тот же класс поправок $O(1/c^2)$ в обычной электродинамике даёт лагранжиан Дарвина: он зависит от скоростей и сохраняет энергию точно. Нужно:
* либо вывести $F_{12}$ строго из действия с плёнкой Ферми (открытая задача 2 в `NOTE_F12…`), и тогда условие Гельмгольца выполнится автоматически, а энергия сохранится;
* либо, если плёнку принципиально нельзя описать действием, явно указать, куда уходит или откуда берётся энергия $W$ за цикл.

**Для вопроса о детальном равновесии.** Динамика плёнки в этом приближении сохраняет фазовый объём и обратима во времени ($t\to-t$, $\mathbf a$ не меняется). Значит, канал «негамильтонова динамика $\Rightarrow$ поток» здесь не открывается. Открывается другой, грубее: несохранение энергии.